In [0]:
from pyspark.sql import functions as F

In [0]:
API_KEY = dbutils.secrets.get(scope="tc_02", key="api_key")
API_SECRET = dbutils.secrets.get(scope="tc_02", key="api_secret")
BOOTSTRAP_SERVER = dbutils.secrets.get(scope="tc_02", key="bootstrap_servers")

bootstrap = BOOTSTRAP_SERVER if ":" in BOOTSTRAP_SERVER else f"{BOOTSTRAP_SERVER}:9092"

In [0]:
DB = 'tc02' # catalog
FINAL_SCHEMA = 'testKafka' # origens
FINAL_TABLE = 'origin' # tc02_alunos

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{API_KEY}" password="{API_SECRET}";'
    ),
    "subscribe": "alunos-eventos",
    "startingOffsets": "earliest",
}

df = spark.readStream.format("kafka").options(**kafka_options).load()
df_parsed = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp")

checkpoint_path = "/Volumes/tc02/testKafka/checkpoints/alunos_eventos_v3"

query = (
    df_parsed.withColumn("_ingestion_ts", F.current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    # .trigger(processingTime="1 minute")
    .toTable(f"{DB}.{FINAL_SCHEMA}.{FINAL_TABLE}")
)

query.processAllAvailable()
print("Lote processado. Status final:", query.status)

In [0]:
# df = spark.read.table(f"{DB}.{FINAL_SCHEMA}.{FINAL_TABLE}")
# df.display()